# 93. 随机森林

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 8 / 34 步：扩展监督/无监督模型工具箱**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 决策树  →  **本章任务：** 随机森林  →  **下一步：** 梯度提升模型
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

做预测时，单独一棵决策树很容易因为数据里一点的偶然波动就“走错路”，结果时好时坏。



## 本章目标

学完本章，你将能够：

- **理解**：理解「随机森林」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「随机森林」的关键输出指标。
- **迁移**：能把「随机森林」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 93.1 核心概念

**背景引入**：做预测时，单独一棵决策树很容易因为数据里一点的偶然波动就“走错路”，结果时好时坏。随机森林让很多棵随机化出来的树一起投票，把单棵树的冲动平均掉，往往能得到更稳也更准的预测。下一节我们就在真实数据上，直观体会它比单棵树“靠谱”在哪里。

- 多棵低相关树平均可降低方差（打个比方：一个人瞎猜容易偏，叫一大群各看各的角度投票再取平均，就不会被某个人的冲动带跑——这就是“三个臭皮匠”。）
- n_estimators 主要影响稳定性和计算量
- OOB 使用每棵树未抽中的样本评估
- 置换重要性衡量打乱特征后的性能下降


## 93.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 随机森林与 OOB | `forest.score()`、`.fit()` | 在乳腺癌数据上比较测试与袋外分数。 | 认为更多树一定解决所有过拟合 |
| 两种特征重要性 | `pd.Series()`、`pd.concat()`、`compare.sort_values()`、`.head()` | 置换重要性在测试集测量，更直接反映泛化性能依赖。 | 高基数特征下只看 impurity importance |


## 93.3 示例 1：随机森林与 OOB

在乳腺癌数据上比较测试与袋外分数。


<!-- math-foundation:chapter-93 -->
### 数学推导｜随机森林通过多棵树降低方差

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜每棵树给出一个有波动的预测。** 记第 $b$ 棵树为 $\hat f_b(x)$，其方差约为 $\sigma^2$。

**第 2 步｜对 $B$ 棵树求平均。** 回归森林使用 $\bar f_B(x)=\sum_b\hat f_b(x)/B$；分类可先平均各类概率再取最大者。

**第 3 步｜展开平均值的方差。** 若任意两棵树预测的相关系数近似为 $\rho$，则

$$
\operatorname{Var}(\bar f_B)
=\sigma^2\left(\rho+\frac{1-\rho}{B}\right)
$$

**第 4 步｜解释两个调参方向。** 增大 $B$ 只会压低 $(1-\rho)/B$；随机抽样特征与样本的价值，是尽量降低树之间的 $\rho$。若所有树高度相似，树再多也仍受 $\rho\sigma^2$ 限制。

**把上面的关系收束为本章计算式：**

$$
\hat{f}_{RF}(x)=\frac{1}{B}\sum_{b=1}^{B}\hat{f}_b(x)\quad\text{或}\quad \hat{y}=\operatorname{mode}\{\hat{y}_b\}_{b=1}^{B}
$$

**符号解释：** $B$ 是树的数量；回归取平均，分类通常取投票或平均概率。

**代码对应：** 用 `n_estimators` 控制树数，并结合 `max_features`、深度和 OOB/验证结果评估。

**使用边界：** 树之间高度相关时集成收益会下降；更多树主要降低随机波动，不自动解决偏差。


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=82
)
forest = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=3,
    oob_score=True,
    n_jobs=-1,
    random_state=82,
).fit(X_train, y_train)
print(
    "OOB/测试:",
    round(forest.oob_score_, 3),
    round(forest.score(X_test, y_test), 3),
)


**练一练**：只把示例里的树从 `n_estimators=300` 改成 `n_estimators=50`，其余保持不变，看看 OOB 分数和测试分数会怎么变。先想猜猜：树少了，分数会明显下降吗？把 `next_estimators` 填好后运行下面单元格自检。


In [ ]:
# 请在下方填写代码
# 练一练：把上面森林的树从 300 棵改成 50 棵，观察 OOB 与测试分数怎么变化。


In [ ]:
# 完整答案：把树从 300 棵改成 50 棵，观察分数是否稳定
next_estimators = 50

_small_forest = RandomForestClassifier(
    n_estimators=next_estimators,
    min_samples_leaf=3,
    oob_score=True,
    n_jobs=-1,
    random_state=82,
).fit(X_train, y_train)
print(
    "50 棵森林 OOB/测试:",
    round(_small_forest.oob_score_, 3),
    round(_small_forest.score(X_test, y_test), 3),
)
print(
    "300 棵森林 OOB/测试:",
    round(forest.oob_score_, 3),
    round(forest.score(X_test, y_test), 3),
)


## 93.4 示例 2：两种特征重要性

置换重要性在测试集测量，更直接反映泛化性能依赖。


In [ ]:
import pandas as pd
from sklearn.inspection import permutation_importance

impurity = pd.Series(
    forest.feature_importances_, index=X.columns, name="impurity"
)
perm = permutation_importance(
    forest, X_test, y_test, n_repeats=10, random_state=82, n_jobs=-1
)
compare = pd.concat(
    [
        impurity,
        pd.Series(perm.importances_mean, index=X.columns, name="permutation"),
    ],
    axis=1,
)
display(compare.sort_values("permutation", ascending=False).head(10).round(4))


## 93.5 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 93.6 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 93.7 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 93.7.1 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 93.7.2 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 93.8 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 93.8.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 93.9 易错点提醒

- 认为更多树一定解决所有过拟合
- 高基数特征下只看 impurity importance
- 用测试集反复筛特征
- 忽略森林比单树更难解释且占用更多资源


## 93.10 练习与作业

1. 比较 50、150、300 棵树
2. 记录 OOB 和测试得分
3. 观察分数是否趋于稳定

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 93.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“比较 50、150、300 棵树”。
2. **独立完成**：不复制示例代码，完成“记录 OOB 和测试得分”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“观察分数是否趋于稳定”，用一两句话说明你修改了什么。

### 93.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 93.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
forest_rows = []
for n in [50, 150, 300]:
    m = RandomForestClassifier(
        n_estimators=n,
        min_samples_leaf=3,
        oob_score=True,
        n_jobs=-1,
        random_state=82,
    ).fit(X_train, y_train)
    forest_rows.append([n, m.oob_score_, m.score(X_test, y_test)])
practice_result = pd.DataFrame(forest_rows, columns=["trees", "oob", "test"])
display(practice_result.round(3))


## 93.12 小结

理解随机森林如何通过样本和特征随机化集成多棵树，并使用袋外评估与置换重要性诊断模型。


### 93.12.1 你已经掌握

- 训练 RandomForestClassifier
- 理解 bootstrap 与特征子采样
- 使用 OOB 分数
- 比较内置重要性与置换重要性


### 93.12.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 93.12.3 需要注意

- 认为更多树一定解决所有过拟合
- 高基数特征下只看 impurity importance
- 用测试集反复筛特征
- 忽略森林比单树更难解释且占用更多资源


### 93.12.4 完成检查

- [ ] 能够训练 RandomForestClassifier
- [ ] 能够理解 bootstrap 与特征子采样
- [ ] 能够使用 OOB 分数
- [ ] 能够比较内置重要性与置换重要性


### 93.12.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
